In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
import sys
import os
import inspect
import datetime
import logging

In [ ]:
# filesuffix = f'_{datetime.datetime.now():%y%m%d_%H%M}'
# log_file = os.path.join(scriptdir, 'logs', f'{program_name}_{filesuffix}.log')

logging.basicConfig(
    level=logging.INFO,
    format=('%(asctime)s:%(name)s:%(levelname)s:%(funcName)s:%(message)s'), #  + logging.BASIC_FORMAT
    handlers=[
        # logging.FileHandler(log_file),
        logging.StreamHandler(),
    ]
)
logger = logging.getLogger(__name__)
logging.getLogger('ib_insync.objects').setLevel(logging.WARN)


In [ ]:
import math
import numpy as np
np.set_printoptions(precision=3, suppress=True)
from numpy.typing import NDArray
import pandas as pd
import ipywidgets as widgets
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import pathlib
import inspect

In [ ]:
import more_itertools as mit
from more_itertools import windowed
from itertools import pairwise

In [ ]:
search_dirs = ['../../../rlabbe/filterpy', '../../../rlabbe', '../..', '../../..', '..', '../../../jj625/mplfinance/src']
pkg_needed = ['mplfinance', 'eventkit', 'ib_insync', 'filterpy', 'Kalman-and-Bayesian-Filters-in-Python']
pkg_dirs = {}

for pkg in pkg_needed:
    for dir in search_dirs:
        potential_dir = pathlib.Path(dir, pkg).resolve()
        if potential_dir.exists() and potential_dir.is_dir():
            pkg_dirs[pkg] = potential_dir
            break

for pkg, pkg_dir in pkg_dirs.items():
    if str(pkg_dir) not in sys.path:
        sys.path.insert(0, str(pkg_dir))
        print(f"{pkg_dir} added to sys.path")

missing_pkgs = [pkg for pkg in pkg_needed if pkg not in pkg_dirs]
if missing_pkgs:
    print(f"Expected directories not found for: {', '.join(missing_pkgs)}")


In [ ]:
# import ib_insync
from ib_insync.objects import BarDataList, BarData

In [ ]:
import mplfinance as mpf
print(f"module loaded: {inspect.getfile(mpf)}")


In [ ]:
import filterpy
print(f"module loaded: {inspect.getfile(filterpy)}")

In [ ]:
import kf_book
from kf_book import book_plots as bp
print(f"module loaded: {inspect.getfile(kf_book)}")

In [ ]:
# util.logToConsole(logging.INFO)
sym = 'SPY'

date = f'{datetime.datetime.now()+datetime.timedelta(days=-2):%Y%m%d}'
filename = f'{sym}_1m_{date}.csv'
bars_df = pd.read_csv(rf'.\data\{filename}', parse_dates=['date'])
if 'open_' in bars_df.columns:
    bars_df.rename(columns={'open_': 'open'}, inplace=True)
if 'timestamp' in bars_df.columns:
    bars_df.drop(columns=['timestamp'], inplace=True)
bars_df['log_avg'] = np.log(bars_df['average'])

In [ ]:
# Check for zeros in OHLC columns
ohlc_columns = ['open', 'high', 'low', 'close', 'average']
zero_ohlc = bars_df[ohlc_columns].eq(0).any()

if zero_ohlc.any():
	print("Zeros found in OHLC columns:")
	print(zero_ohlc[zero_ohlc].index.tolist())
else:
	print("No zeros found in OHLC columns.")


In [ ]:
bars_df.shape

In [ ]:
# Filter bars_df for the time range from 9:30am to 4:00pm
m1 = (bars_df['date'].dt.time >= datetime.time(9, 30)) & (bars_df['date'].dt.time < datetime.time(16, 0))
m2 = (bars_df['date'].dt.time >= datetime.time(9, 30)) & (bars_df['date'].dt.time <= datetime.time(12, 5))
# filtered_bars_df = bars_df[m1]

# Create the date range index
# idx = pd.date_range(start=filtered_bars_df['date'].min(), end=filtered_bars_df['date'].max(), freq='1min')


In [ ]:
def low_to_high_inflection_point(bars: BarDataList, n=15, m=5) -> bool:
    """check if there is a low to high inflection point in the last n bars.
    This is a sign of reversal. We want to catch the reversal early.
    Define inflection point as any of the last m bars lows is lower than any of the previous n-m bars lows
    """
    if len(bars) < n:
        # the first n bars ...
        # case in point: AMD 10/29/2024 9:35 and 9:36 bars
        if len(bars) > 3:
            lows = [b.low for b in bars] # low so far
            lastn_lows = lows[-3:] # last 3 lows
            if min(lastn_lows) == min(lows):
                pass
                # logger.info(f"low_to_high_inflection_point: last 3 lows are the same: {lows} {lastn_lows}")
                # return True
        return False
    lows = [b.low for b in bars[-n:]]
    nplows = bars.low_prices[:bars._npidx][-n:]
    if not np.isclose(lows, nplows, atol=1e-6).all():
        logger.warning(f"lows != nplows: {lows} != {nplows}")
    if min(lows[:-m]) > min(lows[-m:]):
        # logger.info(f"low_to_high_inflection_point: {lows[:-m]} > {lows[-m:]}")
        return True
    return False # no inflection point


In [ ]:
logging.getLogger('low_to_high_inflection_point').setLevel(logging.WARN)

In [ ]:
def low_to_high_inflection_point_new(bars: BarDataList, n=15, m=3) -> bool:
    """check if there is a low to high inflection point in the last n bars.
    an interval contains an inflection point the points before and after it are higher the farther away from the inflection point.
    if plotted, the shape of the interval looks like a 'V' or 'U'
    the eligible inflection points are bars[-m], bars[-m-1], bars[-m-2]
    """
    if len(bars) < n:
        return False

    lows = [b.low for b in bars[-n:]]
    # lows = bars.low_prices[:bars._npidx][-n:]
    eligible_inflection_points = lows[-m-2:-m+1]

    for i in range(len(eligible_inflection_points)):
        if min(lows[:n-m-2+i]) > eligible_inflection_points[i] < min(lows[-m+i+1:]):
            return True

    return False


In [ ]:
with pd.option_context('display.max_columns', 200, 'display.width', 200):
    display(bars_df[m1].head(), bars_df[m1].tail())

In [ ]:
# BarDataList(durationStr='1 D', barSizeSetting='1 min', useRTH=False, from_df=bars_df)

In [ ]:
bars = BarDataList(durationStr='1 D', barSizeSetting='5 secs', useRTH=True, from_df=bars_df[m1])

In [ ]:
len(bars.npdate)

In [ ]:
vlines_10_5 = []
vlines_6_3 = []
vlines_8_3 = []
vlines_15_new = []
df=bars_df[m1].loc[:, ['date', 'open', 'high', 'low', 'close', 'average', 'volume']]

for i in range(1, len(bars)+1):
    # bars = BarDataList(durationStr='1 D', barSizeSetting='5 secs', useRTH=True, from_df=df.iloc[:i])
    bars_subset = bars[:i]
    # if low_to_high_inflection_point(bars, 10, 5):
    #     vlines_10_5.append(df.date.iloc[i-1])
    # if low_to_high_inflection_point(bars, 6, 3):
    #     vlines_6_3.append(df.date.iloc[i-1])
    # if low_to_high_inflection_point(bars, 8, 3):
    #     vlines_8_3.append(df.date.iloc[i-1])
    if low_to_high_inflection_point_new(bars_subset, 15):
        vlines_15_new.append(bars[i-1].date)
        # axs[0].axvline(x=bars[-1].date, color='blue', linestyle='--', linewidth=1)
        # logger.info(f"not inflection point at {bars[-1].date}")

len(vlines_10_5), len(vlines_6_3), len(vlines_8_3), len(vlines_15_new)

In [ ]:
mpf.original_flavor = True
df=bars_df[m1].set_index('date').loc[:, ['open', 'high', 'low', 'close', 'average', 'volume']]
base = df['close'].iloc[0]
df[['open', 'high', 'low', 'close', 'average']] = df[['open', 'high', 'low', 'close', 'average']].apply(lambda x: (x / base) - 1)
df['date'] = df.index

ref = base
def price2pct(px):
    return (px - ref) / ref
def pct2price(pct):
    return pct * ref + ref
fig, axs = mpf.plot(df[:200], style='charles', figratio=(5,1), returnfig=True, vlines=dict(vlines=vlines_15_new[:10], linestyle='--', linewidths=0.8, alpha=0.25, colors='b'))
# axs[0].set_facecolor('#767676') # rgba(160, 160, 160, 0.74) 4C4C4C
secax = axs[0].secondary_yaxis('left', functions=(pct2price, price2pct))
# axs[0].yaxis.set_major_locator(ticker.MultipleLocator(0.01))
axs[0].yaxis.set_major_locator(ticker.MultipleLocator(0.0025))
axs[0].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=2))
axs[0].xaxis.set_major_locator(ticker.MultipleLocator(10))

# for i in range(1, len(df)+1):
#     bars = BarDataList(durationStr='1 D', barSizeSetting='1 min', useRTH=True, from_df=bars_df[m1].iloc[:i])
#     if low_to_high_inflection_point(bars, 15, 10):
#         # axs[0].axvline(x=bars[-1].date, color='blue', linestyle='--', linewidth=1)
#         # logger.info(f"not inflection point at {bars[-1].date}")
#         pass
# axs[0].axvline(x=df.date.iloc[10], color='blue', linestyle='--', linewidth=11)
plt.show()

In [ ]:
import copy
from numpy.random import randn
from scipy.linalg import block_diag
import filterpy
from filterpy.kalman import IMMEstimator, KalmanFilter
from filterpy.common import Q_discrete_white_noise, Q_continuous_white_noise, Saver


In [ ]:
inspect.getfile(filterpy)

In [ ]:
def turning_target(N=600, turn_start=400):
    """ simulate a moving target"""

    #r = 1.
    dt = 1.
    phi_sim = np.array(
        [[1, dt, 0, 0],
         [0, 1, 0, 0],
         [0, 0, 1, dt],
         [0, 0, 0, 1]])

    gam = np.array([[dt**2/2, 0],
                    [dt, 0],
                    [0, dt**2/2],
                    [0, dt]])

    x = np.array([[2000, 0, 10000, -15.]]).T

    simxs = []

    for i in range(N):
        x = np.dot(phi_sim, x)
        if i >= turn_start:
            x += np.dot(gam, np.array([[.075, .075]]).T)
        simxs.append(x)
    simxs = np.array(simxs)

    return simxs


N = 600
dt = 1.
imm_track = turning_target(N)

# create noisy measurements
zs = np.zeros((N, 2))
r = 1
for i in range(N):
    zs[i, 0] = imm_track[i, 0] + randn()*r
    zs[i, 1] = imm_track[i, 2] + randn()*r

ca = KalmanFilter(6, 2)
dt2 = (dt**2)/2
F = np.array([[1, dt, dt2],
              [0,  1,  dt],
              [0,  0,   1]])
            
ca.F = block_diag(F, F)
ca.x = np.array([[2000., 0, 0, 10000, -15, 0]]).T
ca.P *= 1.e-12
ca.R *= r**2
q = np.array([[.05, .125, 1/6],
              [.125, 1/3, .5],
              [1/6, .5, 1]])*1.e-3
ca.Q = block_diag(q, q)
ca.H = np.array([[1, 0, 0, 0, 0, 0],
                 [0, 0, 0, 1, 0, 0]])

# create identical filter, but with no process error
cano = copy.deepcopy(ca)
cano.Q *= 0

filters = [ca, cano]

M = np.array([[0.97, 0.03],
              [0.03, 0.97]])
mu = np.array([0.5, 0.5])
bank = IMMEstimator(filters, mu, M)

xs, probs = [], []
for i, z in enumerate(zs):
    z = np.array([z]).T
    bank.predict()
    bank.update(z)

    xs.append(bank.x.copy())
    probs.append(bank.mu.copy())

xs = np.array(xs)
probs = np.array(probs)
plt.subplot(121)
plt.plot(xs[:, 0], xs[:, 3], 'k')
plt.scatter(zs[:, 0], zs[:, 1], marker='+')

plt.subplot(122)
plt.plot(probs[:, 0])
plt.plot(probs[:, 1])
plt.ylim(-1.5, 1.5)
plt.title('probability ratio p(cv)/p(ca)')
plt.show()

In [ ]:
KalmanFilter(dim_x=3, dim_z=1)

In [ ]:
def make_ca_filter(dt, std_R):
    cafilter = KalmanFilter(dim_x=3, dim_z=1)
    cafilter.x = np.array([0., 0., 0.])
    cafilter.P *= 3
    cafilter.R *= std_R*std_R
    cafilter.Q = Q_discrete_white_noise(dim=3, dt=dt, var=0.02)
    cafilter.F = np.array([[1, dt, 0.5*dt*dt],
                           [0, 1,         dt], 
                           [0, 0,          1]])
    cafilter.H = np.array([[1., 0, 0]])
    return cafilter

def make_cv_filter(dt, std_R):
    cvfilter = KalmanFilter(dim_x=2, dim_z=1)
    cvfilter.x = np.array([0., 0.])
    cvfilter.P *= 3
    cvfilter.R *= std_R*std_R
    cvfilter.F = np.array([[1, dt],
                           [0,  1]], dtype=float)
    cvfilter.H = np.array([[1, 0]], dtype=float)
    cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=0.02)
    return cvfilter

def initialize_const_accel(kf, std_R=None):
    kf.x = np.array([0., 0., 0.])
    kf.P = np.eye(kf.dim_x) * 3
    if std_R is not None:
        kf.R = np.eye(kf.dim_z) * std_R

In [ ]:
Q_discrete_white_noise(dim=3, dt=dt, var=1.)

In [ ]:
Q_continuous_white_noise(dim=3, dt=dt, spectral_density=1.)

In [ ]:
dt = 1.
N = 390
# create noisy measurements
zs = np.zeros((N, 2))
r = 0.5
for i in range(N):
    zs[i, 0] = bars_df[m1].iloc[i]['close']
    zs[i, 1] = bars_df[m1].iloc[i]['average']

# ca = KalmanFilter(3, 1)
ca = make_ca_filter(dt, std_R=0.6) # about 0.1% of the price
s = Saver(ca)
# dt2 = 0.5*dt*dt # (dt**2)/2
# F = np.array([[1, dt, dt2],
#               [0,  1,  dt],
#               [0,  0,   1]])
            
# ca.F = F # block_diag(F, F)
ca.x = np.array([[588., 0, 0]]).T
ca.P *= 1.e-12
ca.R *= r**2
# q = np.array([[.05, .125, 1/6],
#               [.125, 1/3, .5],
#               [1/6, .5, 1]])*1.e-3
# q = Q_discrete_white_noise(dim=3, dt=dt, var=0.1)
q = Q_continuous_white_noise(dim=3, dt=dt, spectral_density=50.)
ca.Q = q # block_diag(q, q)
# ca.H = np.array([[1, 0, 0]])

# create identical filter, but with no process error
cano = copy.deepcopy(ca)
cano.Q *= 0

filters = [ca, cano]

M = np.array([[0.97, 0.03],
              [0.03, 0.97]])
mu = np.array([0.5, 0.5])
bank = IMMEstimator(filters, mu, M)

# xs, probs = [], []
# for i, z in enumerate(zs):
#     z = np.array([z]).T
#     bank.predict()
#     bank.update(z)

#     xs.append(bank.x.copy())
#     probs.append(bank.mu.copy())

# xs = np.array(xs)
# probs = np.array(probs)


In [ ]:
kxs2, a1, b1, c1 = ca.batch_filter(zs[:, 0], saver=s)
# kxs3, a2, b2, c2 = ca.batch_filter(zs[:, 1])

In [ ]:
s.R

In [ ]:
c1

In [ ]:
kxs2[:, 2]

In [ ]:
kxs2[:,1]

In [ ]:
with bp.figsize(30,9):
    bp.plot_track(zs[:, 0], dt=dt)
    bp.plot_filter(kxs2[:, 0], dt=dt, label='KF', alpha=0.6)

    plt.show()

    # add two subplots below the main plot and plot the velocity and acceleration
    # fig = plt.gcf()
    # ax2 = fig.add_subplot(312)
    # ax3 = fig.add_subplot(313)
    plt.plot(kxs2[:, 1], label='velocity', alpha=0.6)
    plt.plot(kxs2[:, 2], label='acceleration', alpha=0.6)

    plt.show()

In [ ]:
# bp.plot_track(zs[:, 1], dt=dt)
# bp.plot_filter(kxs3[:, 0], dt=dt, label='KF')
# plt.show()